# Training wording versus development wording — saved mixed adapter

Check whether the saved mixed-mode model recalls facts with the exact training questions but struggles with held-out wording. Use all 84 paired fact questions in answer-only and thinking modes; repeat the previous development/utility outputs as controls. **346 predictions, zero training updates.** Neither the original nor mixed pilot is relabelled as passed.

Run this local notebook in VS Code on the existing Kaggle GPU kernel with HF_TOKEN attached. It uses evaluation helpers already published at commit `7d4b28f` and embeds the fixed config, so no new GitHub push is needed before executing it. The notebook is orchestration around existing evaluators, not a new trainer. The private adapter is restored by immutable revision and checked against hashes.


In [1]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='wording-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
subprocess.run(['git', 'checkout', '--detach', '7d4b28f1d18d543ea0daacf8f09cb2df9dbf0eed'], cwd=repo_dir, check=True)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
# Earlier notebooks may have imported src from another temporary clone.
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
print('Helper source commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


Cloning into '/kaggle/working/wording-source-iexut69p/repo'...


Helper source commit: 7d4b28f1d18d543ea0daacf8f09cb2df9dbf0eed


HEAD is now at 7d4b28f Preserve thinking-mode results and prepare mixed-mode tiny M1 retry


In [2]:
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], check=True)


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
2.11.0+cu130 5.20.2 0.21.2


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], returncode=0)

In [3]:
import hashlib, json
from src.training.data import select_pilot, read_rows, utility_examples
from src.training.mixed_data import mixed_examples
from src.training.tiny_m1 import evaluate, write_rows
from src.evaluation.mode_diagnostic import SAMPLING, sampled_evaluation, metrics
from src.evaluation.utility_diagnostic import compare_reference
from src.m0_smoke_test import write_json, installed_packages, package_version

# Embedded versioned config: no newly published runner/config is needed remotely.
config = {'run_type': 'training_wording_diagnostic_v1',
 'helper_git_commit': '7d4b28f1d18d543ea0daacf8f09cb2df9dbf0eed',
 'model_id': 'Qwen/Qwen3-0.6B',
 'model_revision': 'c1899de289a04d12100db370d81485cdf75e47ca',
 'training_config': 'configs/training/m1_tiny_mixed_v1.json',
 'training_config_sha256': '48c6314c7067a7b483079df299c9dd7d6acce6e9cf43d86e6a71a7a2765c0f03',
 'batch_size': 16,
 'seed': 42,
 'sampled_max_new_tokens': 1024,
 'artifact': {'repo_id': 'tharundevulapalli/persistent-llm-unlearning-checkpoints',
              'revision': '25764276e3e83e6c20519429854e15216f57e676',
              'path': 'diagnostics/m1-tiny-mixed-v1/m1-tiny-mixed-v1-_avmkde0',
              'evidence_sha256': {'summary.json': '953a51b123dd6b7c78055fcf98d71e70e3ee7abc2fbb021ed501bc5eed0f7ad0',
                                  'selected_train.jsonl': 'd7a84f32178fc3299a5ce1c4fc7493e6640a28de0866a4ad44c11f7e855447bc',
                                  'selected_dev.jsonl': '47e18fc280aa1c81cbaed4dc5677ba5f2dcc4fe0cef8c484d2b8685086b64f4b',
                                  'post_dev_answer_only.jsonl': '20c2eeb72645f0823237d6488391c04be18ef40eb565b54e4202786625c4e51d',
                                  'post_dev_thinking.jsonl': 'adb2fb57db7e5e97d7d2e5396be7a5bfbd74d8b6869297c58bf1f7b800b232b6',
                                  'post_utility_thinking.jsonl': '880e0960759bd770ad8cc31bd79f16b5f6fbc6c45aca9b2896baa15f30e60e6d'}}}
assert hashlib.sha256(Path(config['training_config']).read_bytes()).hexdigest() == config['training_config_sha256']
training_config = json.loads(Path(config['training_config']).read_text())
assert training_config['model_id'] == config['model_id'] and training_config['model_revision'] == config['model_revision']
train_questions, dev_questions = select_pilot(training_config)
expected_mixed, expected_dev = mixed_examples(training_config)
key = lambda row: (row['record_id'], row['task_type'])
train_by_key = {key(row): row for row in train_questions}
assert len(train_by_key) == len(train_questions) == len(dev_questions) == 84
assert set(train_by_key) == {key(row) for row in dev_questions}
train_questions = [train_by_key[key(row)] for row in dev_questions]
for train_row, dev_row in zip(train_questions, dev_questions):
    assert train_row['answer'] == dev_row['answer'] and train_row['question'] != dev_row['question']
    assert train_row['prompt_partition'] == 'm1_train' and dev_row['prompt_partition'] == 'm1_dev'
assert not ({r['template_id'] for r in train_questions} & {r['template_id'] for r in dev_questions})
output_dir = Path(tempfile.mkdtemp(prefix='m1-wording-probe-', dir=remote_root))
config_digest = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
source_files = ['src/training/data.py', 'src/training/mixed_data.py', 'src/training/tiny_m1.py', 'src/evaluation/answers.py', 'src/evaluation/mode_diagnostic.py', 'src/evaluation/utility_diagnostic.py', 'src/m0_knowledge_audit.py']
plan = {'config': config, 'config_canonical_json_sha256': config_digest,
        'source_git_commit': config['helper_git_commit'],
        'source_sha256': {name: hashlib.sha256(Path(name).read_bytes()).hexdigest() for name in source_files},
        'training_updates': 0, 'model_executed': False, 'planned_predictions': 346,
        'training_questions': 84, 'development_questions': 84, 'utility_questions': 10,
        'ordering': 'Pair record/task in original development order; fixed matching batch positions.',
        'arms': {'answer_only': training_config['evaluation'], 'thinking': {'sampling': SAMPLING, 'max_new_tokens': 1024, 'batch_size': 16, 'seed': 42}},
        'scope': 'Generation on training wording is diagnostic evaluation only, not training or held-out performance.'}
write_json(output_dir / 'plan.json', plan)
write_rows(output_dir / 'selected_training_questions.jsonl', train_questions)
write_rows(output_dir / 'selected_development_questions.jsonl', dev_questions)
print('Outputs:', output_dir)
print('84 paired questions x two wordings x two modes + ten utility controls = 346 predictions; zero training updates.')


Outputs: /kaggle/working/m1-wording-probe-hlq0680r
84 paired questions x two wordings x two modes + ten utility controls = 346 predictions; zero training updates.


In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
artifact = config['artifact']
api = HfApi(token=hf_token)
assert api.repo_info(artifact['repo_id'], repo_type='model').private
restore_dir = Path(tempfile.mkdtemp(prefix='wording-restore-', dir=remote_root))
verified = {}
def restore_file(name, expected):
    file = Path(hf_hub_download(repo_id=artifact['repo_id'], filename=artifact['path'] + '/' + name, revision=artifact['revision'], token=hf_token, local_dir=restore_dir))
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name
    verified[name] = expected
    return file
references = {name: restore_file(name, expected) for name, expected in artifact['evidence_sha256'].items()}
original = json.loads(references['summary.json'].read_text())
assert original['plan']['config'] == training_config
assert read_rows(references['selected_train.jsonl']) == expected_mixed
assert read_rows(references['selected_dev.jsonl']) == expected_dev
manifest = original['artifact_files_sha256']
assert set(manifest) == {'adapter/README.md', 'adapter/adapter_config.json', 'adapter/adapter_model.safetensors'}
for name, expected in manifest.items():
    restore_file(name, expected)
adapter_path = restore_dir / artifact['path'] / 'adapter'
adapter_config = json.loads((adapter_path / 'adapter_config.json').read_text())
assert adapter_config['base_model_name_or_path'] == config['model_id'] and adapter_config['revision'] == config['model_revision']
write_json(output_dir / 'source_artifact_verification.json', {'artifact': artifact, 'verified_sha256': verified})
print('Saved mixed adapter, original inputs and reference reports restored with matching hashes. Token is not displayed.')


summary.json:   0%|          | 0.00/31.8k [00:00<?, ?B/s]

selected_train.jsonl:   0%|          | 0.00/134k [00:00<?, ?B/s]

selected_dev.jsonl:   0%|          | 0.00/44.8k [00:00<?, ?B/s]

post_dev_answer_only.jsonl:   0%|          | 0.00/84.1k [00:00<?, ?B/s]

post_dev_thinking.jsonl:   0%|          | 0.00/200k [00:00<?, ?B/s]

post_utility_thinking.jsonl:   0%|          | 0.00/15.2k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

diagnostics/m1-tiny-mixed-v1/m1-tiny-mix(…): reconstructing file:   0%|          |  0.00B / 20.2MB            

diagnostics/m1-tiny-mixed-v1/m1-tiny-mix(…): downloading bytes:           |  0.00B            

Saved mixed adapter, original inputs and reference reports restored with matching hashes. Token is not displayed.


In [5]:
import gc, time
from datetime import datetime, timezone
import torch
from peft import PeftModel
from peft.import_utils import is_torchao_available
from transformers import AutoModelForCausalLM, AutoTokenizer

is_torchao_available()
assert torch.cuda.is_available()
tokenizer = AutoTokenizer.from_pretrained(config['model_id'], revision=config['model_revision'], padding_side='left', token=hf_token)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(config['model_id'], revision=config['model_revision'], dtype=torch.float16, token=hf_token).to('cuda:0')
model = PeftModel.from_pretrained(base, adapter_path, is_trainable=False).eval()
model.requires_grad_(False)
write_json(output_dir / 'generation_metadata.json', {'eos_token_id': model.generation_config.eos_token_id, 'pad_token_id': tokenizer.pad_token_id, 'think_open_token_id': tokenizer.convert_tokens_to_ids('<think>'), 'think_close_token_id': tokenizer.convert_tokens_to_ids('</think>'), 'chat_template_sha256': hashlib.sha256(tokenizer.chat_template.encode()).hexdigest()})
results, controls = {}, {}
sampled_settings = {'batch_size': 16, 'seed': 42, 'sampled_max_new_tokens': 1024}
started = datetime.now(timezone.utc).isoformat()
start_clock = time.perf_counter()
try:
    # Reproduce all original development outputs before testing new training-wording generations.
    results['development_answer_only'] = evaluate(model, tokenizer, dev_questions, training_config)
    write_rows(output_dir / 'development_answer_only.jsonl', results['development_answer_only'])
    results['development_thinking'] = sampled_evaluation(model, tokenizer, dev_questions, True, sampled_settings, output_dir / 'development_thinking.jsonl')
    results['utility_thinking'] = sampled_evaluation(model, tokenizer, utility_examples(), True, sampled_settings, output_dir / 'utility_thinking.jsonl')
    for name, filename, expected_inputs in [('development_answer_only', 'post_dev_answer_only.jsonl', dev_questions), ('development_thinking', 'post_dev_thinking.jsonl', dev_questions), ('utility_thinking', 'post_utility_thinking.jsonl', utility_examples())]:
        previous = read_rows(references[filename])
        actual = results[name]
        assert len(previous) == len(actual) == len(expected_inputs)
        for old, new, example in zip(previous, actual, expected_inputs):
            assert all(old[key] == value and new[key] == value for key, value in example.items())
        different = [row['example_id'] for row, old in zip(actual, previous) if row['raw_response'] != old['raw_response']]
        controls[name] = {'raw_predictions_identical': not different, 'different_example_ids': different}
    write_json(output_dir / 'reference_reproduction.json', controls)
    if not all(c['raw_predictions_identical'] for c in controls.values()):
        write_json(output_dir / 'summary.json', {'execution_complete': False, 'training_updates': 0, 'canonical_m1': False, 'plan': plan, 'controls': controls, 'stop_reason': 'Saved development/utility responses did not reproduce; diagnose environment/loading before wording comparison.'})
        raise RuntimeError('Reference reproduction failed. No training-wording inference started; preserve the evidence.')
    results['training_answer_only'] = evaluate(model, tokenizer, train_questions, training_config)
    write_rows(output_dir / 'training_answer_only.jsonl', results['training_answer_only'])
    results['training_thinking'] = sampled_evaluation(model, tokenizer, train_questions, True, sampled_settings, output_dir / 'training_thinking.jsonl')
    summary = {'execution_complete': True, 'model_executed': True, 'training_updates': 0, 'canonical_m1': False, 'original_pilot_result_unchanged': True, 'plan': plan, 'controls': controls, 'groups': {name: metrics(rows) for name, rows in results.items()}, 'prediction_files_sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in output_dir.glob('*.jsonl') if p.name not in {'selected_training_questions.jsonl', 'selected_development_questions.jsonl'}}, 'started_at_utc': started, 'finished_at_utc': datetime.now(timezone.utc).isoformat(), 'seconds': round(time.perf_counter() - start_clock, 4), 'gpu': torch.cuda.get_device_name(0), 'versions': {p: package_version(p) for p in ['torch', 'transformers', 'peft', 'accelerate']}, 'interpretation': 'Training-wording accuracy is a memorization diagnostic, not held-out performance. One sampled seed; original failed gates remain unchanged.'}
    write_json(output_dir / 'summary.json', summary)
    (output_dir / 'environment.freeze.txt').write_text('\n'.join(installed_packages()) + '\n')
finally:
    del model, base
    gc.collect()
    torch.cuda.empty_cache()


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Thinking=True: generated 16/84 predictions
Thinking=True: generated 32/84 predictions
Thinking=True: generated 48/84 predictions
Thinking=True: generated 64/84 predictions
Thinking=True: generated 80/84 predictions
Thinking=True: generated 84/84 predictions
Thinking=True: generated 10/10 predictions
Thinking=True: generated 16/84 predictions
Thinking=True: generated 32/84 predictions
Thinking=True: generated 48/84 predictions
Thinking=True: generated 64/84 predictions
Thinking=True: generated 80/84 predictions
Thinking=True: generated 84/84 predictions


In [6]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete'] and summary['training_updates'] == 0
print('Reference responses reproduced:', summary['controls'])
paired = {}
for mode in ['answer_only', 'thinking']:
    training_rows = results['training_' + mode]
    development_rows = results['development_' + mode]
    counts = {'correct_both': 0, 'training_only_correct': 0, 'development_only_correct': 0, 'wrong_both': 0}
    pairs = []
    for train_row, dev_row in zip(training_rows, development_rows):
        assert key(train_row) == key(dev_row) and train_row['answer'] == dev_row['answer']
        t, d = train_row['answer_match'], dev_row['answer_match']
        category = 'correct_both' if t and d else 'training_only_correct' if t else 'development_only_correct' if d else 'wrong_both'
        counts[category] += 1
        pairs.append({'record_id': train_row['record_id'], 'task_type': train_row['task_type'], 'entity_partition': train_row['entity_partition'], 'answer': train_row['answer'], 'training_example_id': train_row['example_id'], 'development_example_id': dev_row['example_id'], 'training_question': train_row['question'], 'development_question': dev_row['question'], 'training_raw_response': train_row['raw_response'], 'development_raw_response': dev_row['raw_response'], 'training_correct': t, 'development_correct': d, 'category': category})
    paired[mode] = {'counts': counts, 'pairs': pairs}
    print('\n', mode, '| training wording:', str(summary['groups']['training_' + mode]['answer_matches']) + '/84', '| development wording:', str(summary['groups']['development_' + mode]['answer_matches']) + '/84')
    print('Paired outcomes:', counts)
    for name in ['training_' + mode, 'development_' + mode]:
        group = summary['groups'][name]
        print(name, '| format:', group['format_violations'], '| caps:', group['token_limit_hits'], '| missing boundaries:', group['missing_thinking_boundaries'], '| learning groups:', json.dumps(group['learning_metrics']))
write_json(output_dir / 'paired_wording_review.json', paired)
print('Thinking utility:', str(summary['groups']['utility_thinking']['answer_matches']) + '/10')
print('No training occurred. Review the paired answers before changing training; original mixed-mode pilot remains failed.')


Reference responses reproduced: {'development_answer_only': {'different_example_ids': [], 'raw_predictions_identical': True}, 'development_thinking': {'different_example_ids': [], 'raw_predictions_identical': True}, 'utility_thinking': {'different_example_ids': [], 'raw_predictions_identical': True}}

 answer_only | training wording: 80/84 | development wording: 78/84
Paired outcomes: {'correct_both': 77, 'training_only_correct': 3, 'development_only_correct': 1, 'wrong_both': 3}
training_answer_only | format: 0 | caps: 0 | missing boundaries: 0 | learning groups: {"F/author_book": {"accuracy": 1.0, "correct": 8, "count": 8}, "F/multi_hop": {"accuracy": 1.0, "correct": 12, "count": 12}, "R/author_book": {"accuracy": 1.0, "correct": 16, "count": 16}, "R/multi_hop": {"accuracy": 0.9166666666666666, "correct": 22, "count": 24}}
development_answer_only | format: 0 | caps: 0 | missing boundaries: 0 | learning groups: {"F/author_book": {"accuracy": 1.0, "correct": 8, "count": 8}, "F/multi_ho

In [7]:
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)


Download from Kaggle Output: /kaggle/working/m1-wording-probe-hlq0680r-evidence.zip
